<a href="https://colab.research.google.com/github/AW-RUcode/AW-RUcode/blob/Week-3/Intro_to_Pytorch_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a target="_blank" href="https://colab.research.google.com/github/AW-RUcode/AW-RUcode/blob/Week-3/Intro_to_Pytorch_2.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

We already used and we'll continue to use frequently the following sub-modules from PyTorch:
- [torch.nn](https://docs.pytorch.org/docs/stable/nn.html): contains classes that can construct neural networks.
- torch.nn.functional: contains various functions for data processing and computation.
- torch.optim: contains optimizers to train neural networks.
- torch.utils.data: contains classes that facilitates loading data.

In [ ]:
import torch
from torch import nn
from torch.nn import functional
from torch import optim
from torch.utils import data

torch.manual_seed(42)

Additionally, we can import some utility packages for the visualisation (from matplotlib) and to generate a comprehensive summary of a PyTorch neural network model (torchinfo).

In [ ]:
!pip install torchinfo

import matplotlib.pyplot as plt
import torchinfo

## Loading Datasets from Pytorch

Besides the modules to build our neural networks, Pytorch library also contains a set of [standard datasets](https://docs.pytorch.org/vision/stable/datasets.html). The fact that those datasets are inside Pytorch is handy and can save us a lot of time when learning how to create our first neural network.

Therefore, let's import the [MNIST](https://en.wikipedia.org/wiki/MNIST_database) dataset. MNIST is a popular dataset on the computer vision community that contains a large number of binary images of handwritten digits. Once loaded, Pytorch allows us to easily split the dataset into the commonly used train and test data splits.

In [ ]:
from torchvision import datasets

# Load MNIST dataset
train_dataset = datasets.MNIST(root='./data', train=True, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, download=True)

# Convert to numpy arrays for inspection and visualization
X_train = train_dataset.data.numpy()
y_train = train_dataset.targets.numpy()
X_test = test_dataset.data.numpy()
y_test = test_dataset.targets.numpy()

We can now look into the shape of the imported data and visualise some examples:

In [ ]:
# Print dataset info
print('Image shape: {0}'.format(X_train.shape[1:]))
print('Total number of training samples: {0}'.format(X_train.shape[0]))
print('Total number of test samples: {0}'.format(X_test.shape[0]))

# Visualization: show N x N grid of images
N = 5
start_val = 0  # starting index
fig, axes = plt.subplots(N, N, figsize=(8, 8))
items = list(range(0, 10))

for row in range(N):
    for col in range(N):
        idx = start_val + row + N * col
        axes[row, col].imshow(X_train[idx], cmap='gray')
        fig.subplots_adjust(hspace=0.5)
        y_target = int(y_train[idx])
        target = str(items[y_target])
        axes[row, col].set_title(target)
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])

plt.show()

We have 60000 training samples and 10000 test samples, where each image has a shape of 28×28 pixels. Visualising the data before deploying any algorithm is always a good idea: it is a quick sanity check that can prevent avoidable mistakes.

## Preprocessing Pytorch Datasets

In this first example, we train a simple model to classify the digits on MNIST dataset. We use only `Linear` layers for classification - maltilayer prceptron (MLP). Linear layers take vectors as inputs, so we have to reshape the images into a 1D array to have a single dimension, and define an architecture that could be used in any 1D data. Let's transform the 2D images into 1D arrays by reshaping tensors.

In [ ]:
X_train_flatten = X_train.reshape(X_train.shape[0], -1)
X_test_flatten = X_test.reshape(X_test.shape[0], -1)

print('New X_train shape: {0}'.format(X_train_flatten.shape))

A standard practise is to normalise the pixel values into the range $[0, 1]$ for training stability.

In [ ]:
X_train_flatten = X_train_flatten.astype('float32')
X_test_flatten = X_test_flatten.astype('float32')
X_train_flatten /= 255.0
X_test_flatten /= 255.0

We train a classifier on a dataset in which every instance is labled, so we need to separate the labels so taht we can use them as target variables. The `y_train` and `y_test` labels have the numerical values belonging to the `X_train` and `X_test` images.

Before training we have to convert all numpy tensors to pytorch tensors.

We will use a Categorical CrossEntropy as the loss function, which accepts class label indices as int64 in Pytorch.

After this, we group the training data inputs and labels using the `DataLoader` function to feed it to our network.

In [ ]:
# Convert data to PyTorch tensors
X_train_tensor = torch.from_numpy(X_train_flatten).float()
X_test_tensor = torch.from_numpy(X_test_flatten).float()
y_train_tensor = torch.from_numpy(y_train).long()
y_test_tensor = torch.from_numpy(y_test).long()

# Create dataset and dataloader
train_dataset = data.TensorDataset(X_train_tensor, y_train_tensor)
train_loader = data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataset = data.TensorDataset(X_test_tensor, y_test_tensor)
test_loader = data.DataLoader(test_dataset, batch_size=32, shuffle=False)

## Model Architecture

We often declare a model using the `Sequential` module in PyTorch, which is a container that allows you to stack layers/modules in order and pass input through them sequentially.

We only have two layers here linear where 784 is input size an 10 is the number of hidden units, and ReLU with their number is the same as in linear usints. nn.ReLU() will automatically apply the $\max\{0,x\}$ operation to all 10 outputs simultaneously.

Why we define the output of our layer to be 10? because that corresponds to the 10 different classes in MNIST.

In [ ]:
model = nn.Sequential(
    nn.Linear(784, 10),
    nn.ReLU(),
)

We now have a simple model ready to go!

Note that the model needs to know the shape of the input data. For this reason, the first layer in the Sequential model needs information about the input shape (the following layers can automatically infer the shape, and you do not need to specify it). In our case, we tell the network that the input size is 784. The 784 size vector comes from the flatten operation of our images (28 x 28). If we were using directly images, the input shape would have been (1, 28, 28), which corresponds to the (depth, width, height) of each digit image.

Let's print the model shape output by passing in a dummy input and viewing the shape of the output:

In [ ]:
# Create dummy input with the same shape as your training data
# For flattened MNIST (28x28), input shape is [batch_size, 784]
dummy_input = torch.randn(1, 784)  # batch size of 1

# Pass it through the model
output = model(dummy_input)

# Print the shape of the output
print("Output shape:", output.shape)

Pytorch automatically handles the connections between layers, so there is no need for us to manually set up anything within the architecture.

To get a summary of the model architecture, we can use the torchinfo package:

In [ ]:
torchinfo.summary(model, input_size=(1, 784))

In our example, the first row of the table has $784\cdot 10+ 10=7850$ parameters, where the first term refers to the connections between input data and neurons,  and the second term to the output bias.

## Training setup

Before training our model, we need to configure the learning process. We must define three important parameters here:

*   The loss function that the model will try to minimise.
*   The optimiser used to minimise the loss function and update the weights of the network.
*   The list of metrics you want the model to compute in every step.
*   The device to carry out the computation (e.g., CPU, GPU, TPU, etc.)

In our example, as we are doing classification, we will define the `categorical_crossentropy` as the loss function, and we will compute the accuracy metric. Many other parameters could be tuned, e.g., learning rate, decay factor, or weight normalisation. We will explore in more detail all of these parameters later. For now, we will use the default values for them.

We define the output class as maximum value of linear regression reuslt (i.e. max between ourputs of Relu).

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())
device = torch.device('cpu')

def accuracy(output, target):
    preds = output.argmax(dim=1)
    correct = (preds == target).sum().item()
    return correct / target.size(0)

## Training

To train the model, we must define our training loop. We have to define the number of total epochs the model is going to train. Within the loop, we set the model into training mode, perform a forward pass, compute the loss, perform a backwards pass and log any metrics we desire. We are finally ready to start the learning of our classifier!

In [ ]:
model = model.to(device)

# Training loop
epochs = 10
train_losses = []
train_accuracies = []

for epoch in range(epochs):
    model.train()                             # Set model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()                # Clear gradients
        outputs = model(inputs)               # Forward pass
        loss = criterion(outputs, labels)     # Compute loss
        loss.backward()                       # Backward pass
        optimizer.step()                      # Update weights

        # Metrics
        running_loss += loss.item()
        predicted = outputs.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    accuracy = correct / total
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {running_loss:.4f} - Accuracy: {accuracy:.4f}")

    train_losses.append(running_loss)
    train_accuracies.append(accuracy)


Great, we've trained our first model in PyTorch!

Now imagine we don’t know how many epochs are needed for the model to converge. In Keras, we’d use callbacks, like `EarlyStopping` or `ModelCheckpoint`. PyTorch doesn’t have built-in callbacks, but we can achieve the same functionality manually or with helper libraries like `torchmetrics`, `pytorch_lightning`, or custom logic inside our training loop.

Most importantly, to track training progress, we can store the accuracy and loss values at each epoch and plot them manually.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 6))

axes[0].plot(train_accuracies, label='Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_ylabel('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(train_losses, 'g', label='Loss')
axes[1].set_title('Model Loss')
axes[1].set_ylabel('Cross Entropy Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()

fig.subplots_adjust(hspace=0.5)
plt.tight_layout()
plt.show()

## Evaluating Model

Finally, we check the metrics of our model on the test data by using the method `.evaluate()`:

In [ ]:
# Evaluation in PyTorch
model.eval()  # Set model to evaluation mode

correct = 0
total = 0
loss_total = 0.0

with torch.no_grad():  # Avoid computing gradient for efficiency
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss_total += loss.item()
        predicted = torch.argmax(outputs, dim=1)
        correct += (predicted == yb).sum().item()
        total += yb.size(0)

avg_loss = loss_total / len(test_loader)
accuracy = correct / total

print('Test loss:', avg_loss)
print('Test accuracy:', accuracy)

# Training a Simple Multi-layer Perceptron

Earlier, we showed how to create a simple network that maps directly from $784$ (input size) to $10$ (output size). In the following sections, we explain the basics of creating more complex models, models that combine different sequential layers to perform more accurate classifications.

Hence, we start by defining a network with an extra layer and study how that affects the final test accuracy.

In [ ]:
# Define the model
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(784, 100),
            nn.ReLU(),
            nn.Linear(100, 10),
        )

    def forward(self, x):
        return self.model(x)

# Instantiate model
model2 = SimpleMLP()

# Print model summary (input size is (batch_size, 784))
torchinfo.summary(model2, input_size=(1, 784))

We have increased the number of parameters from 7850 to 79510 by adding this layer. Remember that the number of units given to the last layer defines the dimensionality of the output space, thus, we need the last layer to have the same size that the total number of classes to classify.

Note that we use ReLU (Rectified Linear Unit) activation function after the first dense layer. ReLU is a common activation function, we will give more details about it in future tutorials. However, for now, you can learn more about ReLU [here](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html).

We train our new model and visualise the model curves:

In [ ]:
# Instantiate loss, optimizer, device
model2 = SimpleMLP()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model2.parameters(), lr=0.001)
device = torch.device('cpu')

# Prepare DataLoader
train_dataset = data.TensorDataset(
    torch.tensor(X_train_flatten, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long),
)
train_loader = data.DataLoader(train_dataset, batch_size=32, shuffle=True)

# Train model and record history
epochs = 10
train_loss_history = []
train_acc_history = []

for epoch in range(epochs):
    model2.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        outputs = model2(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        predicted = torch.argmax(outputs, dim=1)
        correct += (predicted == yb).sum().item()
        total += yb.size(0)

    avg_loss = running_loss / len(train_loader)
    accuracy = correct / total

    train_loss_history.append(avg_loss)
    train_acc_history.append(accuracy)

    print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, Accuracy={accuracy:.4f}")

# Plotting accuracy and loss
fig, axes = plt.subplots(2, 1, figsize=(8, 6))

axes[0].plot(train_acc_history)
axes[0].set_title('Model Accuracy')
axes[0].set_ylabel('Accuracy')
axes[0].set_xlabel('Epoch')

axes[1].plot(train_loss_history, 'g')
axes[1].set_title('Model Loss')
axes[1].set_ylabel('Cross Entropy Loss')
axes[1].set_xlabel('Epoch')

fig.subplots_adjust(hspace=0.5)
plt.tight_layout()
plt.show()

We have now trained two different models. Let's plot their metrics to see how that extra layer affected the results in the network performance.

In [ ]:
def evaluate_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        inputs = torch.tensor(X_test, dtype=torch.float32)
        labels = torch.tensor(y_test, dtype=torch.long)
        outputs = model(inputs)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        _, predicted = torch.max(outputs, 1)
        accuracy = (predicted == labels).float().mean().item()
    return loss.item(), accuracy

# Evaluate both models
loss1, acc1 = evaluate_model(model, X_test_flatten, y_test)
loss2, acc2 = evaluate_model(model2, X_test_flatten, y_test)

# Print results
print("Old model:")
print(f"Test loss: {loss1:.4f}")
print(f"Test accuracy: {acc1:.4f}")
print()
print("New model:")
print(f"Test loss: {loss2:.4f}")
print(f"Test accuracy: {acc2:.4f}")

We have improved accuracy results on the test set. Even though there is still margin for parameter tuning, the reached accuracy is already pretty high. We could add more layers or change the number of neurons in each layer to see if we could boost even further the results.

Let's now explore some techniques that will prove useful in the following tutorials.

## Saving/Loading Model

Some networks require long training times (hours, days or even weeks), hence, it is essential to know how to save the models for using them in future times without the need of retraining them every time.

We can save and load the trained model in different ways.

The first way is to save everything into a single PTH file, which will contain:

*   the architecture of the model, allowing to re-create the model
*   the weights of the model
*   the training configuration (loss, optimiser)
*   the state of the optimiser, which permits us to resume training exactly where you left it off.


In [ ]:
# Save model weights and optimizer states if needed
torch.save({
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict(),  # Optional: only needed if resuming training
}, 'my_model.pth')


Once we have a model defined, we can load the weights with the method `load_state_dict(torch.load(file_path))`.:

In [ ]:
model.load_state_dict(torch.load('my_model.pth')['model'])
model.eval()

### Download Models / Save to Google Drive
One of the problems you may face is that the models, or any other file, saved in Colab will not be there permanently. In some cases, you will want to store your model weights in a more lasting way. There are two ways to save your model. First, you can download any file to your computer. To do so, you can use the left-side menu in Colab, and follow the instructions on the image below.

![Screenshot](https://i.ibb.co/yS5JgPd/Screenshot-2021-01-18-at-15-46-04.png)

If you want to use any file you previously downloaded, you will need to manually upload your file using the Colab interface.

Another way to save your weights is by using your Google Drive storage. This can be more convenient, as you can quickly load the models again from your Drive without having to upload manually the file. To do so, you can click on the folder with the Drive symbol highlighted in the following image.

![Screenshot](https://i.ibb.co/PFGw6QR/Screenshot-2021-01-18-at-16-16-44.png)

After clicking on the Drive folder, the code below will appear in your Colab notebook. You need to run it and follow the instructions to have access to your Google Drive folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

After the `drive` folder is mounted, you will see a new folder on your files section in Colab as in the following image.

![Screenshot](https://i.ibb.co/NSM1RFK/Screenshot-2021-01-18-at-16-17-48.png)

Now, you can save the model to your personal Google Drive and also load any model from it.

In [ ]:
torch.save(model.state_dict(), '/content/drive/MyDrive/my_model.pth')
model.load_state_dict(torch.load('/content/drive/MyDrive/my_model.pth'))

Using your Google Drive storage is quite convenient. However, keep in mind that any file saved in your drive will also count towards your Google Drive storage limit.

## Obtaining an Output of an Intermediate Layer

Sometimes we need to check how features of intermediate layers look like. This can be used for extracting features, but also for debugging purposes. That is why we need to see how we can obtain information about any layer within the architecture.

The easiest way is to design a new model that will have as final output our desired layer. We first define the model:

In [ ]:
class CustomMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.first_dense = nn.Linear(784, 64)
        self.second_dense = nn.Linear(64, 128)
        self.final_dense = nn.Linear(128, 10)

    def forward(self, x):
        x1 = functional.relu(self.first_dense(x))   # "first_dense" output
        x2 = functional.relu(self.second_dense(x1)) # "second_dense" output
        x3 = functional.softmax(self.final_dense(x2), dim=1)  # "final_dense" output
        return x1, x2, x3  # return all intermediate activations


In [ ]:
model = CustomMLP()
model.eval()

# Example input
sample_input = torch.randn(1, 784)  # single flattened image
first_out, second_out, final_out = model(sample_input)

print("First Dense Output Shape:", first_out.shape)
print("Second Dense Output Shape:", second_out.shape)
print("Final Output (Softmax):", final_out)

Now, we specify the name of the layer in which we are interested in obtaining the output. In our example, it is the second dense layer ("second_dense"), for which we extract the output from the reconstruction of the network.

In [ ]:
def get_intermediate_output(model, x, layer_name="second_dense"):
    x1 = functional.relu(model.first_dense(x))
    if layer_name == "first_dense":
        return x1
    x2 = functional.relu(model.second_dense(x1))
    if layer_name == "second_dense":
        return x2
    x3 = functional.softmax(model.final_dense(x2), dim=1)
    return x3

## Freezing Layers

Pytorch also allows us to freeze some of the weights of specific layers. Freezing layers means that we can exclude them from training. A typical example where this proved useful is when fine-tuning a model. Remember that when fine-tuning a model, some layers are fixed, and normally only the last ones are trained to finetune the task-specific layers (e.g. classifiers) to the new task.

To freeze layers we can set the requires grad argument (Boolean) of the layer to be non-trainable or trainable:

In [ ]:
# Define a frozen dense layer (Linear layer in PyTorch)
frozen_layer = nn.Linear(in_features=64, out_features=32)

# Freeze it: disable gradient computation
for param in frozen_layer.parameters():
    param.requires_grad = False

In [ ]:
# Instantiate model
model = nn.Sequential(
    nn.Linear(784, 10),
    nn.Softmax(dim=1)
)

print("Summary before freezing:")
print(torchinfo.summary(model, input_size=(1, 784)))

# Freeze all parameters
for param in model.parameters():
    param.requires_grad = False

print("\nSummary after freezing:")
print(torchinfo.summary(model, input_size=(1, 784)))

# Additionally print trainable parameters count and flag for each param
def print_trainable_params(model):
    print("\nParameter trainability:")
    for name, param in model.named_parameters():
        print(f"{name:30} | trainable: {param.requires_grad}")

print_trainable_params(model)

We see in the model description that the frozen model has 0 trainable parameters, while it has 7,850 non-trainable parameters. This fact indicates that we successfully froze that model and the training does not affect it. We can verify this using the results after training both networks, where the trainable model improves accuracy while the frozen one does not change at all.